## Modeling - Hybrid LSTM + MLP (sequential + tabular)

**Architecture**: Two-branch hybrid model.
- **Sequence branch**: LSTM over the 15-frame tracking window (43 features/frame) -> temporal attention (masks padded frames) -> context vector
- **Static branch**: MLP over the tabular features (X_static)
- **Fusion**: concatenated branches -> classification head

**Inputs**:
- `X_seq.npy`    : (N, 15, 43) - left-padded, fixed-constant normalized tracking sequences
- `X_static.npy` : (N, F)      - RAW-magnitude tabular features (scaled per-fold here)
- `y.npy`, `meta.csv` (GAME_ID, EVENT_NUM, SEQ_LEN)

**Scaling**: `X_seq` is already normalized in feature engineering - left untouched. `X_static`
is raw, so a `MinMaxScaler` is fit PER FOLD on the training split inside `run_cv` (no leakage),
matching the tabular model notebooks.

**Phases**: baseline (fixed params) -> tuned (Optuna) -> final (best params, full CV + OOF + interpretability).

**Protocol** (same as the tabular notebooks): every fold trains for a fixed number of epochs (`n_epochs`,
a tuned hyperparameter) with NO early stopping and NO epoch selection, so the validation fold is only used
to score the model, once, at the end of training. Train and validation ROC-AUC give the overfitting gap, and
Optuna maximizes `composite_score = roc_auc_mean - ALPHA * overfit_gap_roc_auc` (trials whose gap exceeds
`MAX_OVERFIT_GAP` score 0.0). The final model is trained on all dev data with the same hyperparameters
(including `n_epochs` and `batch_size`), so it follows exactly the procedure evaluated in CV.

#### **Imports**

In [ ]:
import os
import json
import random
import sys
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import mlflow
import optuna
from optuna.samplers import TPESampler 
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

optuna.logging.set_verbosity(optuna.logging.WARNING)

#### **Config**

In [ ]:
# Run Configuration
MODEL_TYPE     = "HybridLSTM"
MOMENT         = "moment_1"
BALANCED       = True         # this file is the balanced version - see directory name
BASE_TAG       = "balanced" if BALANCED else "unbalanced"
SUFFIX         = f"_{BASE_TAG}"

FRAME_SIZE     = 43  # ball(x,y,z) + 5 OFF(x,y,vx,vy) + 5 DEF(x,y,vx,vy)
MAX_FRAMES     = 15  # left-padded to this length (3s window at 5Hz)
N_FOLDS        = 5
RANDOM_STATE   = 42
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_TRIALS       = 15

ALPHA           = 0.1    # overfitting penalty weight (same as the tabular notebooks)
MAX_OVERFIT_GAP = 0.15   # trials with a larger train-val ROC-AUC gap are scored 0.0

GIT_COMMIT     = "9eb24ae"

DATA_DIR       = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '../../../..', 'data', 'sequences', MOMENT, 'features', 'sequential'))
MLFLOW_DIR     = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '../../../..', 'runs', 'mlruns'))
RUN_DIR        = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '../../../..', 'runs', f"{MODEL_TYPE}_{MOMENT}_{BASE_TAG}"))

# Experiment / run names
EXP_BASELINE   = f"{MODEL_TYPE}_baseline"
EXP_TUNED      = f"{MODEL_TYPE}_tuned"
EXP_FINAL      = f"{MODEL_TYPE}_final"

os.makedirs(MLFLOW_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(os.path.join(MLFLOW_DIR, ".trash"), exist_ok=True)
mlflow.set_tracking_uri(f"file:///{MLFLOW_DIR.replace(os.sep, '/')}")

print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
print(f"Artifacts -> {RUN_DIR}")
print(f"Overfitting - alpha={ALPHA}, hard cap={MAX_OVERFIT_GAP}")


#### **Loading**

In [ ]:
X_seq    = np.load(os.path.join(DATA_DIR, f'X_seq{SUFFIX}.npy'))        # (N, MAX_FRAMES, 43)
X_static = np.load(os.path.join(DATA_DIR, f'X_static{SUFFIX}.npy'))    # (N, F) - raw magnitudes
y        = np.load(os.path.join(DATA_DIR, f'y{SUFFIX}.npy'))           # (N,)
meta     = pd.read_csv(os.path.join(DATA_DIR, f'meta{SUFFIX}.csv'))    # GAME_ID, EVENT_NUM, SEQ_LEN

seq_lens = meta['SEQ_LEN'].values.astype(np.int64)
groups   = meta['GAME_ID'].values          # for game-level CV (StratifiedGroupKFold)
STATIC_DIM = X_static.shape[1]

assert X_seq.shape[1] == MAX_FRAMES, f"X_seq has {X_seq.shape[1]} frames but MAX_FRAMES={MAX_FRAMES} - re-run FE or fix config"
assert X_seq.shape[2] == FRAME_SIZE, f"X_seq frame size {X_seq.shape[2]} != FRAME_SIZE {FRAME_SIZE}"

print(f"X_seq    : {X_seq.shape}")
print(f"X_static : {X_static.shape}")
print(f"y        : {y.shape}  |  class balance (ORB rate): {y.mean():.3f}")
print(f"seq_lens : min={seq_lens.min()}, max={seq_lens.max()}, mean={seq_lens.mean():.1f}")
print(f"STATIC_DIM: {STATIC_DIM}")
print(f"groups   : {len(np.unique(groups))} unique games")

# Input scale check: X_seq should be normalized (~[-1,1]); X_static is raw (scaled per-fold).
print(f"\nX_seq    range [{X_seq.min():.2f}, {X_seq.max():.2f}]  sd {X_seq.std():.3f}  (expect ~[-1,1])")
print(f"X_static range [{X_static.min():.2f}, {X_static.max():.2f}]  sd {X_static.std():.3f}  (raw; scaled per-fold)")
if X_seq.std() > 5:
    print("WARNING: X_seq looks unnormalized - re-run featureEngineering_sequential.")

# Class-balance flag, logged alongside GIT_COMMIT in every mlflow run below
class_counts = pd.Series(y).value_counts()
CLASS_BALANCE_RATIO = class_counts.min() / class_counts.max()
IS_BALANCED = bool(np.isclose(CLASS_BALANCE_RATIO, 1.0, atol=0.02))

# Bundled run metadata, logged with every run (mirrors BASE_PARAMS in the tabular notebooks)
BASE_PARAMS = {
    'model_type': MODEL_TYPE,
    'moment': MOMENT,
    'git_commit': GIT_COMMIT,
    'n_static_features': STATIC_DIM,
    'n_folds': N_FOLDS,
    'cv_strategy': 'StratifiedGroupKFold',
    'random_state': RANDOM_STATE,
    'alpha': ALPHA,
    'max_overfit_gap': MAX_OVERFIT_GAP,
    'class_balanced': IS_BALANCED,
    'loss_pos_weighting': not IS_BALANCED,
}

#### **Dataset**

In [ ]:
class ReboundDataset(Dataset):
    def __init__(self, X_seq, X_static, y, seq_lens):
        self.X_seq    = torch.FloatTensor(X_seq)
        self.X_static = torch.FloatTensor(X_static)
        self.y        = torch.FloatTensor(y)
        self.seq_lens = torch.LongTensor(seq_lens)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.seq_lens[idx], self.X_static[idx], self.y[idx]

#### **Model Architecture**

```
X_seq (B, 15, 43) --> LSTM --> TemporalAttention --> context (B, lstm_hidden)
                                                              |
                                                         concat --> fusion --> logit
                                                              |
X_static (B, F)  --> Linear --> ReLU --> Dropout --> (B, mlp_hidden)
```

**Padding note**: sequences are *left-padded* (zeros at the start, real frames at the end).
The `TemporalAttention` layer receives a boolean mask `pad_mask` (True = padded position)
so it applies `-inf` before softmax, effectively zeroing out padded frames.

In [ ]:
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '../..'))
from model_defs import build_model

#### **Training Utilities**

In [ ]:
EVAL_BATCH_SIZE = 256
METRICS = ["roc_auc", "accuracy", "f1", "precision", "recall"]   # same set as the tabular notebooks


# Seed every RNG the training touches (weight init, dropout, DataLoader shuffling).
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# BCE-with-logits loss. On the balanced branch it is the plain loss. On the unbalanced
# branch (IS_BALANCED is False) positives are weighted by n_neg / n_pos of the TRAINING rows,
# the equivalent of class_weight='balanced' in the tabular models.
def make_criterion(y_train):
    if IS_BALANCED:
        return nn.BCEWithLogitsLoss()
    pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()],
                              dtype=torch.float32, device=DEVICE)
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for x_seq, seq_lens_b, x_static, y_b in loader:
        x_seq, seq_lens_b = x_seq.to(device), seq_lens_b.to(device)
        x_static, y_b     = x_static.to(device), y_b.to(device)

        optimizer.zero_grad()
        logits = model(x_seq, seq_lens_b, x_static)
        loss   = criterion(logits, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # stabilize LSTM training
        optimizer.step()
        total_loss += loss.item() * len(y_b)
    return total_loss / len(loader.dataset)


# P(offensive) for every row in `loader` (eval mode, no shuffling expected).
def predict_proba(model, loader, device):
    model.eval()
    probs = []
    with torch.no_grad():
        for x_seq, seq_lens_b, x_static, _ in loader:
            logits = model(x_seq.to(device), seq_lens_b.to(device), x_static.to(device))
            probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs)


# Same metrics as the tabular SCORING dict (macro-averaged, threshold 0.5).
def compute_metrics(y_true, probs):
    preds = (probs >= 0.5).astype(int)
    return {
        "roc_auc":   roc_auc_score(y_true, probs),
        "accuracy":  accuracy_score(y_true, preds),
        "f1":        f1_score(y_true, preds, average="macro"),
        "precision": precision_score(y_true, preds, average="macro", zero_division=0),
        "recall":    recall_score(y_true, preds, average="macro", zero_division=0),
    }


# StratifiedGroupKFold cross-validation for the HybridModel, following the same
# protocol as the tabular notebooks:
#   - each fold trains for exactly params['n_epochs'] epochs (no early stopping, no
#     best-epoch selection), so the validation fold never influences training;
#   - the model is then scored ONCE on the training portion and ONCE on the
#     validation portion, giving the train/validation pair the overfitting gap needs.
# Splits at the GAME_ID level (via `groups`) so no game contributes events to both
# the train and validation fold.
#
# If collect_oof=True, each fold's result also carries 'val_idx' and 'val_probs'
# (out-of-fold probabilities of that fold's validation rows), for assembling
# oof_predictions.csv in the final-phase cell.
#
# Returns the per-fold results list: [{'train': {...}, 'val': {...}, ...}, ...]
def run_cv(X_seq, X_static, y, seq_lens, groups, params, tag='', collect_oof=False):
    skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_results = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_static, y, groups)):
        set_seed(RANDOM_STATE + fold)
        criterion = make_criterion(y[tr_idx])   # weights computed from the training rows only

        # Per-fold scaling of the STATIC branch: fit on the train split ONLY (no leakage).
        # X_seq is already fixed-constant normalized in feature engineering, so leave it.
        static_scaler = MinMaxScaler().fit(X_static[tr_idx])
        Xs_tr  = static_scaler.transform(X_static[tr_idx])
        Xs_val = static_scaler.transform(X_static[val_idx])

        tr_ds  = ReboundDataset(X_seq[tr_idx],  Xs_tr,  y[tr_idx],  seq_lens[tr_idx])
        val_ds = ReboundDataset(X_seq[val_idx], Xs_val, y[val_idx], seq_lens[val_idx])

        tr_loader      = DataLoader(tr_ds,  batch_size=params['batch_size'], shuffle=True)
        tr_eval_loader = DataLoader(tr_ds,  batch_size=EVAL_BATCH_SIZE)
        val_loader     = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE)

        model = build_model(params, STATIC_DIM, DEVICE)
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr           = params['lr'],
            weight_decay = params.get('weight_decay', 1e-4)
        )

        for epoch in range(params['n_epochs']):
            train_epoch(model, tr_loader, optimizer, criterion, DEVICE)

        tr_probs  = predict_proba(model, tr_eval_loader, DEVICE)
        val_probs = predict_proba(model, val_loader, DEVICE)

        entry = {
            'train': compute_metrics(y[tr_idx],  tr_probs),
            'val':   compute_metrics(y[val_idx], val_probs),
        }
        if collect_oof:
            entry['val_idx']   = val_idx
            entry['val_probs'] = val_probs
        fold_results.append(entry)

        print(f"  {tag} Fold {fold+1}: val AUC={entry['val']['roc_auc']:.4f} | "
              f"train AUC={entry['train']['roc_auc']:.4f} | "
              f"gap={entry['train']['roc_auc'] - entry['val']['roc_auc']:+.4f}")

    return fold_results


# Flat metrics dict with the same keys as the tabular run_cv_and_log:
# - per-metric mean/std/train_mean for every metric in METRICS
# - overfit_gap_<metric> = train_mean - test_mean
# - composite_score = roc_auc_mean - alpha * overfit_gap_roc_auc
def summarize_cv(fold_results):
    metrics = {}
    for metric in METRICS:
        test_scores  = np.array([r['val'][metric]   for r in fold_results])
        train_scores = np.array([r['train'][metric] for r in fold_results])
        test_mean  = float(np.mean(test_scores))
        train_mean = float(np.mean(train_scores))

        metrics[f"{metric}_mean"]       = test_mean
        metrics[f"{metric}_std"]        = float(np.std(test_scores))
        metrics[f"{metric}_train_mean"] = train_mean
        metrics[f"overfit_gap_{metric}"] = train_mean - test_mean

    metrics["composite_score"] = (
        metrics["roc_auc_mean"] - ALPHA * metrics["overfit_gap_roc_auc"]
    )
    return metrics


# Cross-validate and log params + metrics + overfitting diagnostics to MLflow
# (the active run). Returns (metrics, fold_results).
def run_cv_and_log(params, phase=None, tag='', collect_oof=False):
    fold_results = run_cv(X_seq, X_static, y, seq_lens, groups, params,
                          tag=tag, collect_oof=collect_oof)
    metrics = summarize_cv(fold_results)

    mlflow.log_params({**BASE_PARAMS, **params, "phase": phase})
    mlflow.log_metrics(metrics)

    if phase in ("baseline", "final"):
        cv_results = {}
        for metric in METRICS:
            cv_results[f"test_{metric}"]  = [float(r['val'][metric])   for r in fold_results]
            cv_results[f"train_{metric}"] = [float(r['train'][metric]) for r in fold_results]
        cv_path = os.path.join(RUN_DIR, f"cv_results_{phase}.json")
        with open(cv_path, "w") as f:
            json.dump(cv_results, f, indent=2)

    return metrics, fold_results


def print_summary(label: str, metrics: dict) -> None:
    gap = metrics["overfit_gap_roc_auc"]
    print(
        f"{label}\n"
        f"  roc_auc   : {metrics['roc_auc_mean']:.4f} +/- {metrics['roc_auc_std']:.4f}  "
        f"(train {metrics['roc_auc_train_mean']:.4f}, gap {gap:+.4f})\n"
        f"  accuracy  : {metrics['accuracy_mean']:.4f} +/- {metrics['accuracy_std']:.4f}  "
        f"(train {metrics['accuracy_train_mean']:.4f}, gap {metrics['overfit_gap_accuracy']:+.4f})\n"
        f"  composite : {metrics['composite_score']:.4f}  "
        f"[= roc_auc - {ALPHA}*gap]"
    )

#### **Phase 1 - Baseline**
Fixed hyperparameters, including a fixed epoch budget. Confirms the pipeline works and gives an initial benchmark (and a first look at the train-validation gap) before running HPO.

In [ ]:
mlflow.set_experiment(EXP_BASELINE)

BASELINE_PARAMS = {
    'lstm_hidden': 64,
    'lstm_layers': 2,
    'mlp_hidden': 64,
    'fusion_hidden': 64,
    'dropout': 0.3,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'batch_size': 64,
    'n_epochs': 30,
}

with mlflow.start_run(run_name=MODEL_TYPE):
    metrics, _ = run_cv_and_log(BASELINE_PARAMS, phase='baseline', tag='baseline')

print_summary("Baseline", metrics)

#### **Phase 2 - Optuna Hyperparameter Tuning**
**Objective**: `composite_score = roc_auc_mean - alpha * overfit_gap_roc_auc` (same as the tabular notebooks).
Trials where `overfit_gap_roc_auc > MAX_OVERFIT_GAP` are penalized to 0.0.

Each trial runs full CV with a fixed epoch budget. Tuning covers architecture (hidden sizes, layers, dropout), training (lr, weight decay, batch size) and the epoch budget `n_epochs`, which takes the place of early stopping.

In [ ]:
mlflow.set_experiment(EXP_TUNED)

def objective(trial):
    params = {
        'lstm_hidden':   trial.suggest_categorical('lstm_hidden', [32, 64, 128]),
        'lstm_layers':   trial.suggest_int('lstm_layers', 1, 3),
        'mlp_hidden':    trial.suggest_categorical('mlp_hidden', [32, 64, 128]),
        'fusion_hidden': trial.suggest_categorical('fusion_hidden', [32, 64, 128]),
        'dropout':       trial.suggest_float('dropout', 0.1, 0.5),
        'lr':            trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'weight_decay':  trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True),
        'batch_size':    trial.suggest_categorical('batch_size', [32, 64, 128]),
        'n_epochs':      trial.suggest_int('n_epochs', 10, 60, step=5),
    }
    with mlflow.start_run(run_name=f'trial_{trial.number}', nested=True):
        metrics, _ = run_cv_and_log(params, phase='tuned', tag=f't{trial.number}')

        if metrics["overfit_gap_roc_auc"] > MAX_OVERFIT_GAP:
            mlflow.log_metric("pruned_overfit", 1.0)
            return 0.0

        mlflow.log_metric("pruned_overfit", 0.0)
        return metrics["composite_score"]

with mlflow.start_run(run_name=MODEL_TYPE):
    study = optuna.create_study(
        direction = 'maximize',
        study_name=f"{MODEL_TYPE}_{MOMENT}_{BASE_TAG}",
        sampler   = TPESampler(seed=RANDOM_STATE)
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest trial #{study.best_trial.number} - composite_score: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
# Optuna visualization - importance of each hyperparameter
optuna.visualization.matplotlib.plot_param_importances(study)
plt.tight_layout()
plt.show()

optuna.visualization.matplotlib.plot_optimization_history(study)
plt.tight_layout()
plt.show()

In [ ]:
# Inspect overfitting: trials whose gap exceeded the cap were scored 0.0
trials_df = study.trials_dataframe()
valid = trials_df[trials_df["value"] > 0].copy()

print(f"Valid trials (gap <= {MAX_OVERFIT_GAP}): {len(valid)} / {len(trials_df)}")
print(valid[["value"]].describe().T.to_string())

assert len(valid) > 0, "Every trial exceeded MAX_OVERFIT_GAP - best_params would come from a discarded trial. Loosen the search space or the cap."

#### **Phase 3 - Final Evaluation**
Full CV with the best params found by Optuna (same epoch budget and batch size as the tuned trial), followed by the OOF predictions and a final model trained on all dev data with the same hyperparameters.

In [ ]:
mlflow.set_experiment(EXP_FINAL)

BEST_PARAMS = dict(study.best_params)   # includes n_epochs and batch_size; used unchanged below

with mlflow.start_run(run_name=MODEL_TYPE) as final_run:
    metrics, fold_results = run_cv_and_log(BEST_PARAMS, phase='final', tag='final', collect_oof=True)

FINAL_RUN_ID = final_run.info.run_id
print_summary("Final", metrics)

In [ ]:
# Out-of-fold probabilities, used by calibrate.py
oof_val_idx = np.concatenate([f['val_idx'] for f in fold_results])
oof_val_probs = np.concatenate([f['val_probs'] for f in fold_results])

oof_probs = np.empty(len(y))
oof_probs[oof_val_idx] = oof_val_probs  # val_idx across folds covers every row exactly once

oof_df = pd.DataFrame({
    "GAME_ID": meta["GAME_ID"].values,
    "EVENT_NUM": meta["EVENT_NUM"].values,
    "y_true": y,
    "y_prob": oof_probs,
})

oof_path = os.path.join(RUN_DIR, "oof_predictions.csv")
oof_df.to_csv(oof_path, index=False)

with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_artifact(oof_path)

print(f"OOF predictions: {len(oof_df)} rows | true offensive rate {oof_df['y_true'].mean():.3f} | mean predicted prob {oof_df['y_prob'].mean():.3f}")

#### **Interpretability - Attention Weights**
Train one model on the full dataset with best params, then extract attention weights across all samples.
This reveals *which time steps before the anchor point* (shot release for moment_1, rim contact for moment_2) the model focuses on, split by rebound outcome.

In [ ]:
# Final model on all dev data, with the same n_epochs and batch_size as the CV folds.
# Used for the attention/SHAP analysis and the holdout evaluation below.
set_seed(RANDOM_STATE)
static_scaler   = MinMaxScaler().fit(X_static)
X_static_scaled = static_scaler.transform(X_static)
full_ds     = ReboundDataset(X_seq, X_static_scaled, y, seq_lens)
full_loader = DataLoader(full_ds, batch_size=BEST_PARAMS['batch_size'], shuffle=True)

final_model = build_model(BEST_PARAMS, STATIC_DIM, DEVICE)
optimizer   = torch.optim.Adam(final_model.parameters(), lr=BEST_PARAMS['lr'],
                               weight_decay=BEST_PARAMS.get('weight_decay', 1e-4))
criterion   = make_criterion(y)

for epoch in range(BEST_PARAMS['n_epochs']):
    loss = train_epoch(final_model, full_loader, optimizer, criterion, DEVICE)
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d} | loss: {loss:.4f}")

print("Final model trained.")

In [ ]:
# Collect attention weights for every sample
final_model.eval()
all_attn, all_labels = [], []

eval_loader = DataLoader(full_ds, batch_size=128, shuffle=False)
with torch.no_grad():
    for x_seq_b, seq_lens_b, x_static_b, y_b in eval_loader:
        x_seq_b    = x_seq_b.to(DEVICE)
        seq_lens_b = seq_lens_b.to(DEVICE)
        x_static_b = x_static_b.to(DEVICE)
        _, attn_w  = final_model(x_seq_b, seq_lens_b, x_static_b, return_attn=True)
        all_attn.append(attn_w.cpu().numpy())
        all_labels.extend(y_b.numpy())

all_attn   = np.concatenate(all_attn, axis=0)   # (N, MAX_FRAMES)
all_labels = np.array(all_labels)

print(f"Attention weights collected: {all_attn.shape}")

In [ ]:
# Mean attention weight per time step, split by rebound type
# Sequences are left-padded: frame 0 = oldest (up to 3s before the anchor), frame 14 = the anchor
# (shot release for moment_1, rim contact for moment_2)
# Map to seconds: 15 frames at 5Hz = 0.2s per frame -> range [-3s, 0s]
ANCHOR_LABEL = {'moment_1': 'shot release', 'moment_2': 'rim contact'}[MOMENT]
time_axis = np.linspace(-MAX_FRAMES / 5, 0, MAX_FRAMES)

mean_attn_orb = all_attn[all_labels == 1].mean(axis=0)
mean_attn_drb = all_attn[all_labels == 0].mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_axis, mean_attn_orb, label='Offensive rebound', color='tab:orange', linewidth=2)
ax.plot(time_axis, mean_attn_drb, label='Defensive rebound', color='tab:blue',   linewidth=2)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5, label=ANCHOR_LABEL.capitalize())
ax.set_xlabel(f'Time before {ANCHOR_LABEL} (s)')
ax.set_ylabel('Mean attention weight')
ax.set_title('Temporal Attention: Which Frames Drive the Prediction?')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
model_path = os.path.join(RUN_DIR, "final_model.pt")
scaler_path = os.path.join(RUN_DIR, "final_scaler.joblib")

torch.save(final_model.state_dict(), model_path)
joblib.dump(static_scaler, scaler_path)

with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_artifact(model_path)
    mlflow.log_artifact(scaler_path)

print(f"Saved final_model weights -> {model_path}")
print(f"Saved static_scaler       -> {scaler_path}")

#### **Interpretability - SHAP on Static Branch**
Fix the sequence input to the dataset mean (background), vary only `X_static`.
This lets us use `GradientExplainer` to attribute predictions to individual static features.

In [ ]:
# Load static feature names
static_feature_names = pd.read_csv(os.path.join(DATA_DIR, f'static_feature_names{SUFFIX}.csv'), header=None)[0].tolist()
assert len(static_feature_names) == STATIC_DIM, \
    f"Feature name count ({len(static_feature_names)}) does not match STATIC_DIM ({STATIC_DIM})"

In [ ]:
# Wraps HybridModel for SHAP analysis on the static branch only.
# X_seq and seq_lens are fixed to background values;
# only X_static is the varying input.
# Output: sigmoid probability (B, 1) as expected by GradientExplainer.
class StaticWrapper(nn.Module):
    def __init__(self, model, bg_x_seq, bg_seq_lens):
        super().__init__()
        self.model       = model
        self.bg_x_seq    = bg_x_seq    # (1, T, F)
        self.bg_seq_lens = bg_seq_lens # (1,)

    def forward(self, x_static):
        B       = x_static.shape[0]
        x_seq   = self.bg_x_seq.expand(B, -1, -1)
        s_lens  = self.bg_seq_lens.expand(B)
        logits  = self.model(x_seq, s_lens, x_static)
        return torch.sigmoid(logits).unsqueeze(1)   # (B, 1)


# Background sequence = dataset mean; seq_len = median
bg_x_seq    = torch.FloatTensor(X_seq.mean(axis=0, keepdims=True)).to(DEVICE)  # (1, MAX_FRAMES, 43)
bg_seq_lens = torch.LongTensor([int(np.median(seq_lens))]).to(DEVICE)          # (1,)

wrapper = StaticWrapper(final_model, bg_x_seq, bg_seq_lens).to(DEVICE)
wrapper.eval()

# Background sample for GradientExplainer (100 random samples)
rng           = np.random.default_rng(RANDOM_STATE)
bg_idx        = rng.choice(len(X_static), size=100, replace=False)
bg_static     = torch.FloatTensor(static_scaler.transform(X_static[bg_idx])).to(DEVICE)

explainer     = shap.GradientExplainer(wrapper, bg_static)

# Explain 500 samples
sample_idx    = rng.choice(len(X_static), size=500, replace=False)
sample_static = torch.FloatTensor(static_scaler.transform(X_static[sample_idx])).to(DEVICE)
shap_values   = explainer.shap_values(sample_static)   # list of arrays, one per output

print(f"SHAP values shape: {np.array(shap_values).shape}")

In [ ]:
# Beeswarm plot - feature impact on ORB probability
shap_arr = shap_values[0] if isinstance(shap_values, list) else shap_values

shap.summary_plot(
    shap_arr,
    X_static[sample_idx],
    feature_names = static_feature_names,
    show          = True
)

In [ ]:
# Bar plot - mean absolute SHAP (feature importance)
shap.summary_plot(
    shap_arr,
    X_static[sample_idx],
    feature_names = static_feature_names,
    plot_type     = 'bar',
    show          = True
)

#### **Holdout Evaluation**

Scores the `final_model` + `static_scaler` trained above (fit on ALL dev rows,
best params) on the game-level holdout set, and logs the results as `holdout_*`
metrics into the SAME MLflow run as the CV numbers from Phase 3 (`FINAL_RUN_ID`)
- so `roc_auc_mean` (CV) and `holdout_roc_auc` (true holdout) sit side by side
for direct comparison. Requires `featureEngineering_sequential.ipynb` to have
been run with `HOLDOUT=True` for this moment, producing the matching `X_seq_holdout*`, `X_static_holdout*`,
`y_holdout*`, `meta_holdout*` files. Holdout is intentionally **not** balanced - it
reflects the real class distribution, for an honest generalization check.

In [ ]:
HOLDOUT_SUFFIX = "_holdout"

X_seq_holdout    = np.load(os.path.join(DATA_DIR, f'X_seq{HOLDOUT_SUFFIX}.npy'))
X_static_holdout = np.load(os.path.join(DATA_DIR, f'X_static{HOLDOUT_SUFFIX}.npy'))
y_holdout        = np.load(os.path.join(DATA_DIR, f'y{HOLDOUT_SUFFIX}.npy'))
meta_holdout     = pd.read_csv(os.path.join(DATA_DIR, f'meta{HOLDOUT_SUFFIX}.csv'))
seq_lens_holdout = meta_holdout['SEQ_LEN'].values.astype(np.int64)

assert X_static_holdout.shape[1] == STATIC_DIM, \
    f"Holdout static width {X_static_holdout.shape[1]} != training STATIC_DIM {STATIC_DIM}"

# static_scaler was fit on ALL dev X_static above (interpretability cell) - reuse it
# as-is here, never refit on holdout.
X_static_holdout_scaled = static_scaler.transform(X_static_holdout)
holdout_ds     = ReboundDataset(X_seq_holdout, X_static_holdout_scaled, y_holdout, seq_lens_holdout)
holdout_loader = DataLoader(holdout_ds, batch_size=128, shuffle=False)

final_model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for x_seq_b, seq_lens_b, x_static_b, y_b in holdout_loader:
        x_seq_b, seq_lens_b = x_seq_b.to(DEVICE), seq_lens_b.to(DEVICE)
        x_static_b = x_static_b.to(DEVICE)
        logits = final_model(x_seq_b, seq_lens_b, x_static_b)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(y_b.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
all_preds  = (all_probs >= 0.5).astype(int)

# Keys mirror the CV metric names (roc_auc/accuracy/f1/precision/recall) so
# holdout_<key> lines up next to <key>_mean in the same run.
holdout_metrics = {
    "accuracy": accuracy_score(all_labels, all_preds),
    "roc_auc": roc_auc_score(all_labels, all_probs),
    "f1": f1_score(all_labels, all_preds, average="macro"),
    "precision": precision_score(all_labels, all_preds, average="macro"),
    "recall": recall_score(all_labels, all_preds, average="macro"),
}

# Reopen the exact Final run (not a new experiment) so CV and holdout numbers
# live in the same row.
with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_metrics({f"holdout_{k}": v for k, v in holdout_metrics.items()})
    mlflow.log_param("n_holdout", len(y_holdout))

print(f"Holdout: {len(y_holdout)} rows, offensive rate {y_holdout.mean():.3f}")
for k, v in holdout_metrics.items():
    print(f"  holdout_{k:9s}: {v:.4f}")